In [1]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 1.7 MB/s eta 0:00:00


In [4]:
from google.colab import userdata
import os
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] = API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)



# Context Runtime

In [5]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    context_schema=ColourContext
)

In [8]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='078ec81d-6e81-4493-80dc-58d3dcb5f216'),
              AIMessage(content="As an AI, I don't have access to your personal information, so I don't know what your favorite color is!\n\nYou'll have to tell me! What is it?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cd0d4-60ac-7ee3-9d54-1fddbe4df9ce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 221, 'total_tokens': 228, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 181}})]}


# Accessing Context with tool

In [10]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [11]:
agent = create_agent(
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [12]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='07e64fd3-4ce4-446c-a7e3-3f56d52f3b83'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_favourite_colour', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'c737e1d0-bbfe-485a-85f9-3a048dca375e': 'CtEBAb4+9vvTKOP7ckwYRF4a61/kBjNfeU6MXaHhxZ/k6Sq0hjIYhz7xeeuNVL+uJQ39Xvr9tGqMyRRUxTl1O8DRGhKUK6CwIs2u1xU4TciUmD/eoeL3Yf1GtYQ3Wod23W63NSHZ0zRgfFxWW15ZikYdO0ZeDik7m2nxzHvT44O9sszA+iPLSaS6vH1sBixNBk49gW6H0G7zGV6tbLf62DZYUsFNj7gfPXUF/LncpNO8qGB/QYMK0GGLayNBmDEf+0VFBZfImO53uLsdSbhUvjsOxbo='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cd0db-e7ee-7231-b630-e8e70764d158-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'c737e1d0-bbfe-485a-85f9-3a048dca375e', 'type': 'tool_call'}], invalid_tool_calls=[], usage

In [13]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColourContext(favourite_c...vourite_colour='yellow'), input_type=ColourContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='b7e03a14-1dde-4c38-9852-d7a96eba1bc8'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_favourite_colour', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'f306d602-873e-45bc-908b-3ee43a6c8152': 'CtEBAb4+9vsuDHWyAh3Tx9qrr4VAIYEshhHLlkjTHjD+KxnMJkEWTcFXOS6DnoHHlKLTdh4Iq/ca5fNKeGV338vWP5x/obQBekJY9qBv15WvNS8iFyplByZ8RAx323GxE/OIdqAqUnw7ujJIvZABkMpUM+qP5Z4Km2zVPZ0eab7ImYKBBJwGbPztAWUhmtFvj8YgP9T3h40HEOhlzCmbXYyJM48NEp8+trZ5kHTw2C7I7axvH4p3b3LdPwTtJ0AdITV0XIF8xCvqIwVVE0MR/UjD0N8='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cd0de-6463-7271-bca2-c7bb80e4f547-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'f306d602-873e-45bc-908b-3ee43a6c8152', 'type': 'tool_call'}], invalid_tool_calls=[], usage